# RAG System: AWS Customer Agreement Q&A

This notebook walks through the **Retrieval-Augmented Generation (RAG)** pipeline step by step, so you can run each part on its own and see exactly what's happening.

**What this notebook does, in order:**
1. Read the AWS Customer Agreement PDF and pull out the raw text
2. Split that text into overlapping chunks
3. Turn each chunk into an embedding (a list of numbers representing its meaning)
4. Store those embeddings in a FAISS index for fast similarity search
5. Take a question, find the most relevant chunks, and build a prompt
6. Send that prompt to a local LLM (Ollama) to generate the final answer
7. Handle the case where the document doesn't actually contain the answer

**Before running this notebook**, make sure:
- You've run `pip install -r requirements.txt` (see the list in the first code cell, or the backend's requirements.txt)
- [Ollama](https://ollama.com) is installed and running (`ollama serve`), with a model pulled (`ollama pull llama3.2`)
- The PDF file `aws_customer_agreement.pdf` is in the same folder as this notebook (or update the `PDF_PATH` variable below)

This notebook is meant for understanding and experimenting with the RAG logic. The actual submission also includes a FastAPI backend (`main.py`) that wraps this same logic into API endpoints with SQL logging - this notebook does NOT do any database logging, it's just for exploring the pipeline.


## Step 0: Install and import everything we need

In [2]:
# If you haven't installed these yet, uncomment and run this cell once:
# !pip install pypdf sentence-transformers faiss-cpu numpy requests

import json
import os
import numpy as np
import faiss
import requests
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

print("All libraries imported successfully.")

c:\Users\Deekshitha\rag_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully.


## Step 1: Settings

We keep all our "tunable" settings together in one cell, so it's easy to see and change them.

**Why these numbers?**
- `CHUNK_SIZE = 800`: The AWS Agreement has fairly dense legal paragraphs (each numbered section is roughly 500-900 characters). 800 characters usually keeps one full clause/section together in a single chunk.
- `CHUNK_OVERLAP = 150`: About one sentence's worth of overlap, so if an important sentence falls right on a chunk boundary, it still appears completely in at least one chunk.
- `TOP_K = 3`: We retrieve the 3 closest chunks per question. That's roughly 2400 characters of context - enough detail to answer most questions, without overloading the prompt with irrelevant text.
- `MAX_DISTANCE_THRESHOLD = 1.2`: If even the *best* matching chunk is farther than this, we assume the question isn't really about our document, and we say "not found" instead of forcing the LLM to guess.


In [3]:
PDF_PATH = "aws_customer_agreement.pdf"   # change this if your PDF is elsewhere

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"   # small, free, runs locally on CPU

TOP_K = 3
MAX_DISTANCE_THRESHOLD = 1.2   # FAISS L2 distance - LOWER means MORE similar

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2:0.5b"   # must match a model you've pulled with `ollama pull llama3.2`

print("Settings loaded.")

Settings loaded.


## Step 2: Extract text from the PDF

We use `pypdf` to read every page and join all the text together into one big string. This is the simplest possible approach - no special handling of tables/columns, just plain text extraction, which is enough for a contract-style document like this one.


In [4]:
def extract_text_from_pdf(pdf_path):
    """Reads every page of the PDF and returns one big string of text."""
    reader = PdfReader(pdf_path)
    all_text = []

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:   # some pages might come back empty, skip those
            all_text.append(page_text)

    full_text = " ".join(all_text)
    return full_text


full_text = extract_text_from_pdf(PDF_PATH)

print("Total characters extracted:", len(full_text))
print()
print("Preview of the first 500 characters:")
print(full_text[:500])

Total characters extracted: 60501

Preview of the first 500 characters:
AWS Customer Agreement
**For additional information related to each AWS Contracting Party, see the 
.
*Please note that as of August 1, 2025, customers located in Indonesia contract with our Indonesia
based AWS Contracting Party, as provided in Section 12. See the  for more
information.
*Please note that as of January 1, 2026, customers located in Taiwan who have provided a Uniﬁed
Business Number (UBN) and are using the invoicing payment method for their accounts contract with
our Taiwan based A


## Step 3: Split the text into overlapping chunks

Think of the whole document as one long string of characters. We slide a "window" of `CHUNK_SIZE` characters across it, moving forward by `(CHUNK_SIZE - CHUNK_OVERLAP)` each time. That overlap is why neighboring chunks share a bit of text at their edges.

Example with small numbers (`chunk_size=10, overlap=3`) on `"ABCDEFGHIJKLMNOP"`:
- chunk 1: `"ABCDEFGHIJ"` (characters 0-10)
- chunk 2: `"HIJKLMNOPQ"` (characters 7-17, sharing `"HIJ"` with chunk 1)


In [5]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Splits text into overlapping chunks of `chunk_size` characters."""
    chunks = []
    step = chunk_size - overlap

    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:   # skip empty chunks (can happen right at the end)
            chunks.append(chunk)

        start += step

    return chunks


chunks = chunk_text(full_text)

print("Number of chunks created:", len(chunks))

Number of chunks created: 94


In [6]:
# Let's look at the first two chunks to SEE the overlap for ourselves
print("--- Chunk 0 (last 200 characters) ---")
print(chunks[0][-200:])
print()
print("--- Chunk 1 (first 200 characters) ---")
print(chunks[1][:200])
print()
print("Notice the text at the end of Chunk 0 reappears at the start of Chunk 1 - that's our overlap.")

--- Chunk 0 (last 200 characters) ---
s of June 1, 2026, all customers located in Mexico contract with our Mexico based
AWS Contracting Party, as provided in Section 12. See the   for more information.
Last Updated: June 01, 2026
This AWS

--- Chunk 1 (first 200 characters) ---
contract with our Mexico based
AWS Contracting Party, as provided in Section 12. See the   for more information.
Last Updated: June 01, 2026
This AWS Customer Agreement (this “Agreement”) contains the

Notice the text at the end of Chunk 0 reappears at the start of Chunk 1 - that's our overlap.


## Step 4: Generate embeddings for every chunk

An embedding is just a list of numbers (a "vector") that represents the *meaning* of a piece of text. Chunks with similar meaning end up with similar numbers, which is what lets us later search "which chunk is closest in meaning to this question?"

We use `sentence-transformers` with the `all-MiniLM-L6-v2` model - it's small, free, and runs entirely on your own computer (no API key, no internet needed after the first download).

**Note:** the very first time you run this, the model will be downloaded automatically (a few hundred MB). After that it's cached locally and loads instantly.


In [7]:
print("Loading the embedding model (only needs internet the very first time)...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Model loaded.")

Loading the embedding model (only needs internet the very first time)...


c:\Users\Deekshitha\rag_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Deekshitha\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 834.34it/s]


Model loaded.


In [8]:
# This turns our list of chunk texts into a 2D array of numbers:
# one row per chunk, each row has 384 numbers (that's how many dimensions
# this particular model uses to represent meaning).
chunk_embeddings = embedding_model.encode(chunks, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")   # FAISS expects float32

print("Embeddings shape:", chunk_embeddings.shape)
print("(That's", chunk_embeddings.shape[0], "chunks, each represented by", chunk_embeddings.shape[1], "numbers.)")

Batches: 100%|██████████| 3/3 [00:02<00:00,  1.12it/s]

Embeddings shape: (94, 384)
(That's 94 chunks, each represented by 384 numbers.)


## Step 5: Build a FAISS index

FAISS is a library for fast similarity search over vectors. We use `IndexFlatL2`, the simplest kind of FAISS index:

- "Flat" means it compares the query against every single chunk (a "brute force" search). That's perfectly fine here since we only have a small number of chunks - no need for a fancier approximate index.
- "L2" means it measures plain Euclidean (straight-line) distance between two vectors. **Lower distance = more similar meaning.**


In [9]:
dimension = chunk_embeddings.shape[1]   # 384 for this model

index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("FAISS index built. It currently holds", index.ntotal, "chunk vectors.")

FAISS index built. It currently holds 94 chunk vectors.


In [10]:
# Optional: save the index and chunks to disk, so we don't have to
# re-embed everything every time we restart this notebook.

faiss.write_index(index, "faiss_index.bin")

with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f)

print("Saved faiss_index.bin and chunks.json to the current folder.")

Saved faiss_index.bin and chunks.json to the current folder.


In [11]:
# To reload them later (e.g. in a new notebook session), you'd run:
#
# index = faiss.read_index("faiss_index.bin")
# with open("chunks.json") as f:
#     chunks = json.load(f)
#
# This skips re-extracting the PDF and re-computing embeddings from scratch.

## Step 6: Retrieve the most relevant chunks for a question

Now for the actual "Retrieval" part of RAG. Given a question, we:
1. Embed the question the same way we embedded the chunks
2. Ask FAISS for the `TOP_K` closest chunk vectors
3. Check whether the *best* match is actually close enough to be considered relevant - if the question is totally unrelated to the document, even the "best" match will be far away


In [12]:
def retrieve_relevant_chunks(query, index, chunks, top_k=TOP_K, threshold=MAX_DISTANCE_THRESHOLD):
    """
    Returns (retrieved_chunks, is_relevant).
    retrieved_chunks is a list of dicts: {"chunk_id": int, "text": str, "distance": float}
    is_relevant tells us whether the best match is close enough to trust.
    """
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, top_k)
    distances, indices = distances[0], indices[0]   # unwrap, since we only searched 1 query

    retrieved = []
    for distance, idx in zip(distances, indices):
        if idx == -1:   # FAISS returns -1 if there are fewer chunks than top_k
            continue
        retrieved.append({"chunk_id": int(idx), "text": chunks[idx], "distance": float(distance)})

    if not retrieved:
        return [], False

    best_distance = retrieved[0]["distance"]
    is_relevant = best_distance <= threshold

    return retrieved, is_relevant

In [13]:
# Try it out with a question that SHOULD be in the document
test_question = "How are fees and charges billed?"

retrieved, is_relevant = retrieve_relevant_chunks(test_question, index, chunks)

print("Question:", test_question)
print("Is relevant:", is_relevant)
print()
for r in retrieved:
    print(f"--- Chunk {r['chunk_id']} (distance: {r['distance']:.3f}) ---")
    print(r["text"][:200], "...")
    print()

Question: How are fees and charges billed?
Is relevant: True

--- Chunk 12 (distance: 0.804) ---
dd new fees and charges for any existing Services you are using by giving you at least
30 days’ prior notice. We may elect to charge you interest at the rate of 1.5% per month (or the highest
rate per ...

--- Chunk 13 (distance: 0.957) ---
other additions
thereto) that are imposed on that party upon or with respect to the transactions and payments under
this Agreement. All fees payable by you are exclusive of Indirect Taxes, except wher ...

--- Chunk 74 (distance: 0.980) ---
). You will only be liable to pay the INR Equivalent Fees
indicated in each invoice.
We will invoice you from our registered oﬃce at the address of your establishment (as registered with
the tax autho ...



In [14]:
# Now try a question that is clearly NOT about this document
test_question_2 = "What's the weather like today?"

retrieved_2, is_relevant_2 = retrieve_relevant_chunks(test_question_2, index, chunks)

print("Question:", test_question_2)
print("Is relevant:", is_relevant_2)
print("Best match distance:", retrieved_2[0]["distance"] if retrieved_2 else "N/A")
print()
print("Since is_relevant is False, we'll skip calling the LLM entirely for this one,")
print("and just respond with 'I could not find an answer to this in the document.'")

Question: What's the weather like today?
Is relevant: False
Best match distance: 1.884779930114746

Since is_relevant is False, we'll skip calling the LLM entirely for this one,
and just respond with 'I could not find an answer to this in the document.'


## Step 7: Build the prompt

We take the retrieved chunks and the user's question, and assemble them into one prompt for the LLM. The instructions are deliberately strict:

- "Use ONLY the context below" - tells the model not to rely on outside/world knowledge
- An exact fallback sentence to use if the answer isn't in the context - this is our main defense against hallucination, since we're not just hoping the model decides to be honest, we're telling it exactly what to say.


In [15]:
def build_prompt(question, context_chunks):
    """Builds the final prompt string we send to the LLM."""
    context_text = "\n\n".join([f"[Chunk {c['chunk_id']}]: {c['text']}" for c in context_chunks])

    prompt = f"""You are a helpful assistant that answers questions about the AWS Customer Agreement.

Use ONLY the context below to answer the question. Do not use any outside knowledge.
If the answer is not contained in the context, reply exactly with: "I could not find an answer to this in the document."

Context:
{context_text}

Question: {question}

Answer:"""

    return prompt


# Let's see what the prompt looks like for our first test question
example_prompt = build_prompt(test_question, retrieved)
print(example_prompt[:800])
print("... (truncated for display)")

You are a helpful assistant that answers questions about the AWS Customer Agreement.

Use ONLY the context below to answer the question. Do not use any outside knowledge.
If the answer is not contained in the context, reply exactly with: "I could not find an answer to this in the document."

Context:
[Chunk 12]: dd new fees and charges for any existing Services you are using by giving you at least
30 days’ prior notice. We may elect to charge you interest at the rate of 1.5% per month (or the highest
rate permitted by law, if less) on all late payments. If we suspend your account under Section 4.1 or
terminate your use of the Services pursuant to Section 5.2(b)(ii), we may elect not to bill you for fees
and charges after suspension unless your account is reinstated.
3.2 Taxes. Each party w
... (truncated for display)


## Step 8: Call Ollama to generate the answer

[Ollama](https://ollama.com) lets us run an open-source LLM locally, for free, with no API key. Before running the next cell:

1. Install Ollama from https://ollama.com
2. In a terminal, run: `ollama serve` (starts the local server)
3. In another terminal, run: `ollama pull llama3.2` (downloads the model, once)

If Ollama isn't running, the cell below will raise a connection error - that's expected, just start Ollama and try again.


In [16]:
def call_ollama(prompt, model=OLLAMA_MODEL, url=OLLAMA_URL):
    """Sends the prompt to a locally running Ollama server and returns the generated text."""
    response = requests.post(
        url,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False   # we want the whole answer back at once, not streamed piece by piece
        },
        timeout=60
    )
    response.raise_for_status()
    result = response.json()
    return result.get("response", "").strip()


# Try it on our example prompt from Step 7
answer = call_ollama(example_prompt)
print("Question:", test_question)
print("Answer:", answer)

Question: How are fees and charges billed?
Answer: Fees and charges will be charged based on the terms and conditions of the Services, including taxes imposed by applicable laws. The amount to be charged for each Service is determined by the relevant provider's fee schedule or at the time of service delivery. For example, under certain circumstances, fees may include a certain percentage of the total cost or a fixed charge per month as per Section 4.1. If services are suspended or terminated based on legal requirements or exceptions specified in Section 5.2(b)(ii), interest will be charged at the rate of 1.5% per month.


## Step 9: Put it all together - the full RAG pipeline

This function wraps every step above into one call: retrieve -> (maybe skip LLM if irrelevant) -> build prompt -> call Ollama -> return the answer with its sources.

This is the same logic used inside the FastAPI backend's `/ask` endpoint - the notebook version here is for exploring/demoing it interactively.


In [17]:
def answer_question(question, index, chunks):
    """
    Runs the full RAG pipeline for one question.
    Returns (answer_text, source_chunks, answer_found).
    """
    retrieved_chunks, is_relevant = retrieve_relevant_chunks(question, index, chunks)

    if not is_relevant:
        # No point calling the LLM on chunks that aren't actually related
        return "I could not find an answer to this in the document.", [], False

    prompt = build_prompt(question, retrieved_chunks)
    answer = call_ollama(prompt)

    # The model itself might also say it doesn't know, even with context -
    # we check its wording too so our answer_found flag stays accurate.
    answer_found = "could not find an answer" not in answer.lower()

    sources = [{"chunk_id": c["chunk_id"], "text": c["text"]} for c in retrieved_chunks]

    return answer, sources, answer_found

In [18]:
# Try the full pipeline on a few different questions
demo_questions = [
    "What happens if I breach my payment obligations?",
    "Can AWS terminate my account immediately?",
    "What's the capital of France?",   # clearly out of scope, should say "not found"
]

for q in demo_questions:
    answer, sources, found = answer_question(q, index, chunks)
    print("Q:", q)
    print("A:", answer)
    print("Answer found in document:", found)
    print("Number of source chunks used:", len(sources))
    print("-" * 80)

Q: What happens if I breach my payment obligations?
A: If you are in material breach of this Agreement and your payment obligations under Section 3 do not exceed the amounts paid by AWS for the Services that gave rise to the liability during the 12-month period before the liability arose, then nothing in this section will limit any service credits under this agreement or any other payment obligations under this agreement.
Answer found in document: True
Number of source chunks used: 3
--------------------------------------------------------------------------------
Q: Can AWS terminate my account immediately?
A: No, you cannot immediately stop using the Amazon Web Services services by being in breach of your obligations under this Agreement.
Answer found in document: True
Number of source chunks used: 3
--------------------------------------------------------------------------------
Q: What's the capital of France?
A: I could not find an answer to this in the document.
Answer found in do

## Next steps

This notebook covers the core RAG logic (retrieval + generation). The full assignment also needs:

- A **FastAPI backend** that exposes this logic as `POST /ingest`, `POST /ask`, and `GET /analytics` endpoints
- **SQL logging** of every question asked (question text, whether an answer was found, response time) so we can compute usage analytics
- A **React frontend** with a chat-style interface and an analytics dashboard

Those pieces live in the `backend/` and `frontend/` folders of the full project - this notebook is just the "engine" at the center of it all, pulled out so it's easier to read, run, and explain step by step.
